In [ ]:
import pandas as pd
import numpy as np
from sklearn.utils import shuffle
from sklearn.metrics import accuracy_score
from tqdm import tqdm
from epsilon_greedy_bandit import *
from linUCB import *
from thompson_sampling import *

In [2]:
# Load the dataset
df = pd.read_csv('warfarin_one_hot_encoded_full_power.csv')
# Get the number of arms from the number of unique labels
n_arms = df['Therapeutic Dose of Warfarin'].nunique()
df.shape, n_arms


((5528, 168), 3)

# epsilon greedy bandit

In [5]:
# Set up the simulation
n_iterations = 20
cumulative_regrets = []
average_accuracies = []


for i in range(n_iterations):
    # Shuffle the dataset
    df_shuffled = shuffle(df, random_state=i)
    
    # Split the data into features and labels
    X = df_shuffled.drop('Therapeutic Dose of Warfarin', axis=1).values
    y = df_shuffled['Therapeutic Dose of Warfarin'].values
    
    # Initialize the bandit
    bandit = EpsilonGreedyBandit(n_arms)
    
    # Track performance
    correct_predictions = 0
    cumulative_regret = 0
    
    # Simulate the bandit selecting arms and receiving reward
    for j in range(len(df_shuffled)):
        # The bandit makes a prediction (selects an arm)
        chosen_arm = bandit.select_arm()
        
        # Get the actual label
        actual_label = y[j]
        
        # Check if the bandit's prediction was correct
        if chosen_arm == actual_label:
            reward = 1
            correct_predictions += 1
        else:
            reward = 0
            
        # Update the bandit with the reward for the chosen arm
        bandit.update(chosen_arm, reward)
        
        # Calculate regret (difference between the optimal and chosen action)
        optimal_reward = 1  # assuming the optimal action would always be correct
        regret = optimal_reward - reward
        cumulative_regret += regret
    
    # Store the cumulative regret for this iteration
    cumulative_regrets.append(cumulative_regret)
    
    # Calculate accuracy for this iteration
    accuracy = correct_predictions / len(df_shuffled)
    average_accuracies.append(accuracy)
    
    print(f"Iteration {i+1}: Accuracy = {accuracy}, Cumulative Regret = {cumulative_regret}")

# Calculate the average accuracy over all iterations
final_average_accuracy = np.mean(average_accuracies)
print(f"Average Accuracy over {n_iterations} iterations: {final_average_accuracy}")



Iteration 1: Accuracy = 0.5806801736613604, Cumulative Regret = 2318
Iteration 2: Accuracy = 0.5774240231548481, Cumulative Regret = 2336
Iteration 3: Accuracy = 0.5839363241678727, Cumulative Regret = 2300
Iteration 4: Accuracy = 0.5877351664254703, Cumulative Regret = 2279
Iteration 5: Accuracy = 0.5734442836468886, Cumulative Regret = 2358
Iteration 6: Accuracy = 0.5830318379160637, Cumulative Regret = 2305
Iteration 7: Accuracy = 0.5779667149059334, Cumulative Regret = 2333
Iteration 8: Accuracy = 0.5794138929088278, Cumulative Regret = 2325
Iteration 9: Accuracy = 0.5786903039073806, Cumulative Regret = 2329
Iteration 10: Accuracy = 0.5743487698986975, Cumulative Regret = 2353
Iteration 11: Accuracy = 0.5738060781476122, Cumulative Regret = 2356
Iteration 12: Accuracy = 0.5833936324167872, Cumulative Regret = 2303
Iteration 13: Accuracy = 0.5828509406657019, Cumulative Regret = 2306
Iteration 14: Accuracy = 0.578328509406657, Cumulative Regret = 2331
Iteration 15: Accuracy = 0.583

# lin UCB

In [3]:

# Define parameters
alpha = 1.0  # Exploration parameter
n_iterations = 20
cumulative_regrets = []
average_accuracies = []

# Get the number of arms and features
n_arms = df['Therapeutic Dose of Warfarin'].nunique()
n_features = df.shape[1] - 1  # Number of features is total columns minus the label column

for i in range(n_iterations):
    df_shuffled = shuffle(df, random_state=i)
    X = df_shuffled.drop('Therapeutic Dose of Warfarin', axis=1).values
    y = df_shuffled['Therapeutic Dose of Warfarin'].values
    
    # Initialize the LinUCB model
    linucb = LinUCB(alpha, n_arms, n_features)
    
    correct_predictions = 0
    cumulative_regret = 0

    for j in tqdm(range(len(df_shuffled)), desc=f'Iteration {i+1}'):
        x = X[j]
        chosen_arm = linucb.select_arm(x)
        actual_label = y[j]
        
        reward = 1 if chosen_arm == actual_label else 0
        correct_predictions += reward
        
        linucb.update(chosen_arm, x, reward)
        
        optimal_reward = 1  # assuming the optimal action would always be correct
        regret = optimal_reward - reward
        cumulative_regret += regret
    
    cumulative_regrets.append(cumulative_regret)
    
    accuracy = correct_predictions / len(df_shuffled)
    average_accuracies.append(accuracy)
    
    print(f"Iteration {i+1}: Accuracy = {accuracy}, Cumulative Regret = {cumulative_regret}")

final_average_accuracy = np.mean(average_accuracies)
print(f"Average Accuracy over {n_iterations} iterations: {final_average_accuracy}")


Iteration 1: 100%|██████████| 5528/5528 [00:52<00:00, 105.81it/s]


Iteration 1: Accuracy = 0.6338639652677279, Cumulative Regret = 2024


Iteration 2: 100%|██████████| 5528/5528 [00:51<00:00, 107.52it/s]


Iteration 2: Accuracy = 0.6293415340086831, Cumulative Regret = 2049


Iteration 3: 100%|██████████| 5528/5528 [00:53<00:00, 103.06it/s]


...

Iteration 18: Accuracy = 0.6414616497829233, Cumulative Regret = 1982


Iteration 19: 100%|██████████| 5528/5528 [00:54<00:00, 101.54it/s]


Iteration 19: Accuracy = 0.6333212735166426, Cumulative Regret = 2027


Iteration 20: 100%|██████████| 5528/5528 [01:01<00:00, 90.53it/s] 

Iteration 20: Accuracy = 0.6400144717800289, Cumulative Regret = 1990
Average Accuracy over 20 iterations: 0.6353020984081043


# Thompson sampling

In [ ]:
# Define parameters
n_iterations = 20
cumulative_regrets = []
average_accuracies = []

# Get the number of arms from the number of unique labels
n_arms = df['Therapeutic Dose of Warfarin'].nunique()

for i in range(n_iterations):
    df_shuffled = shuffle(df, random_state=i)
    X = df_shuffled.drop('Therapeutic Dose of Warfarin', axis=1).values
    y = df_shuffled['Therapeutic Dose of Warfarin'].values
    
    # Initialize the Thompson Sampling bandit
    ts_bandit = ThompsonSamplingBandit(n_arms)
    
    correct_predictions = 0
    cumulative_regret = 0
    
    # Simulate the decision process
    for j in range(len(df_shuffled)):
        chosen_arm = ts_bandit.select_arm()
        actual_label = y[j]
        
        reward = 1 if chosen_arm == actual_label else 0
        correct_predictions += reward
        
        ts_bandit.update(chosen_arm, reward)
        
        optimal_reward = 1  # assuming the optimal action would always be correct
        regret = optimal_reward - reward
        cumulative_regret += regret
    
    cumulative_regrets.append(cumulative_regret)
    
    accuracy = correct_predictions / len(df_shuffled)
    average_accuracies.append(accuracy)
    
    print(f"Iteration {i+1}: Accuracy = {accuracy}, Cumulative Regret = {cumulative_regret}")

final_average_accuracy = np.mean(average_accuracies)
print(f"Average Accuracy over {n_iterations} iterations: {final_average_accuracy}")